# Populate the Exact-Greedy Sub-300 Archive

Generate a low-score K43 archive using the simplest classical baseline: for every available edge, calculate its exact immediate monochromatic-K5 score reduction and flip the best edge. Tabu memory allows the search to leave local minima. No danger shaping, neural policy, or macro action is used.

Seeds are organized into live archive cycles. Each cycle freezes and shuffles the best eligible archive pool, consumes every seed once, then refreshes from SQLite. Descendants created by one cycle can therefore become ancestors for the next cycle. Only distinct best colorings with exact score 299 or lower are archived.

In [1]:
# Imports and Project Root

from pathlib import Path
from time import perf_counter

import numpy as np

from ramsey import (
    RArchiveBatch,
    RArchiveBatchConfig,
    RArchiveCycleConstruction,
    REnvironment,
    REnvironmentConfig,
    RGraph,
    RGreedyPolicy,
    RMonochromaticObjective,
    RProblem,
    RSearch,
    RSQLiteArchive,
    RTabuMemory,
    RTabuMemoryConfig,
)

project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

if not (project_root / "ramsey").is_dir():
    raise RuntimeError(
        "Run this notebook from the RamseyNumber root "
        "or notebooks directory."
    )

ImportError: cannot import name 'RArchiveCycleConstruction' from 'ramsey' (C:\code\RamseyNumber\ramsey\__init__.py)

In [ ]:
# Batch Configuration

RANDOM_SEED = 202_608_061
N_VERTICES = 43

RUN_NAME = "greedy-exact-sub-300-001"
TARGET_SUB_300_COLORINGS = 500
MAXIMUM_ATTEMPTS = 1_000
START_ITERATION = 0

TARGET_MAXIMUM_SCORE = 299
ARCHIVE_SEED_SCORE_LIMIT = 399
MINIMUM_ARCHIVE_SEEDS = 100
ACTIVE_SEED_POOL_SIZE = 250

SEARCH_STEPS = 500
EDGE_TABU_TENURE = 20
VISITED_STATE_WINDOW = 2_000
REPORT_INTERVAL = 10

DATABASE_PATH = (
    project_root
    / "data"
    / "ramsey_colorings.sqlite3"
)

In [ ]:
# Runtime, Graph, and Existing Archive

rng = np.random.default_rng(
    RANDOM_SEED
)

graph = RGraph(
    RProblem.r55(
        n_vertices=N_VERTICES,
    )
)

existing_archive = globals().get("archive")

if existing_archive is not None:
    existing_archive.close()

archive = RSQLiteArchive(
    DATABASE_PATH
)

eligible_seed_count = (
    archive.coloring_count_in_score_range(
        maximum_score=ARCHIVE_SEED_SCORE_LIMIT,
        graph=graph,
    )
)

existing_target_count = (
    archive.coloring_count_in_score_range(
        maximum_score=TARGET_MAXIMUM_SCORE,
        graph=graph,
    )
)

if eligible_seed_count < MINIMUM_ARCHIVE_SEEDS:
    archive.close()

    raise RuntimeError(
        f"Expected at least {MINIMUM_ARCHIVE_SEEDS} "
        f"archived sub-{ARCHIVE_SEED_SCORE_LIMIT + 1} seeds; "
        f"found {eligible_seed_count}."
    )

print("Database:", DATABASE_PATH.resolve())
print("Archive best:", archive.best_score(graph))
print("Eligible archived seeds:", eligible_seed_count)
print("Existing sub-300 colorings:", existing_target_count)
print("Target sub-300 colorings:", TARGET_SUB_300_COLORINGS)

In [ ]:
# Live Cyclic Seed Queue

construction = RArchiveCycleConstruction(
    archive=archive,
    rng=rng,
    maximum_score=ARCHIVE_SEED_SCORE_LIMIT,
    limit=ACTIVE_SEED_POOL_SIZE,
)

cycle_size = construction.start_cycle(graph)

print("Construction:", construction.name)
print("Active seed pool size:", cycle_size)
print("Current cycle:", construction.cycle_number)
print("Each completed cycle refreshes from the live archive.")

In [ ]:
# Bare Exact-Greedy Tabu Search

objective = RMonochromaticObjective()

memory = RTabuMemory(
    number_of_edges=graph.number_of_edges,
    config=RTabuMemoryConfig(
        edge_tenure=EDGE_TABU_TENURE,
        visited_state_window=VISITED_STATE_WINDOW,
    ),
)

environment = REnvironment(
    graph=graph,
    objective=objective,
    memory=memory,
    config=REnvironmentConfig(
        max_steps=SEARCH_STEPS,
        use_aspiration=True,
    ),
)

policy = RGreedyPolicy(
    rng=rng,
    use_objective_reward=False,
)

search = RSearch(
    environment=environment,
    policy=policy,
)

print("Objective:", objective.name)
print("Policy:", policy.name)
print("Search steps per attempt:", SEARCH_STEPS)
print("Danger shaping: disabled")

In [ ]:
# Assemble the Strict Sub-300 Archive Batch

batch = RArchiveBatch(
    graph=graph,
    construction=construction,
    search=search,
    archive=archive,
)

batch_config = RArchiveBatchConfig(
    run_name=RUN_NAME,
    target_count=TARGET_SUB_300_COLORINGS,
    maximum_attempts=MAXIMUM_ATTEMPTS,
    maximum_score=TARGET_MAXIMUM_SCORE,
    start_iteration=START_ITERATION,
    record_steps=False,
    save_out_of_range=False,
)

print("Run name:", batch_config.run_name)
print("Maximum attempts:", batch_config.maximum_attempts)
print("Only unique sub-300 best results will be archived.")

In [ ]:
# Progress Observer

progress = {
    "start": perf_counter(),
    "best": archive.best_score(graph),
}

def report_attempt(attempt_result):
    result = attempt_result.search_result

    new_database_best = (
        attempt_result.archive_record is not None
        and (
            progress["best"] is None
            or result.best_score < progress["best"]
        )
    )

    if new_database_best:
        progress["best"] = result.best_score

    should_report = (
        attempt_result.attempt % REPORT_INTERVAL == 0
        or attempt_result.in_score_range
        or new_database_best
    )

    if not should_report:
        return

    elapsed = perf_counter() - progress["start"]

    flags = []

    if attempt_result.in_score_range:
        flags.append("SUB-300")

    if attempt_result.new_unique_coloring:
        flags.append("NEW-UNIQUE")

    if new_database_best:
        flags.append("NEW-BEST")

    flag_text = (
        " | " + " | ".join(flags)
        if flags
        else ""
    )

    print(
        f"Attempt {attempt_result.attempt:5d} | "
        f"cycle={construction.cycle_number:2d} | "
        f"source={attempt_result.construction_name:24s} | "
        f"initial={result.initial_score:4d} | "
        f"final={result.final_score:4d} | "
        f"best={result.best_score:4d} | "
        f"eligible={attempt_result.eligible_count:4d}"
        f"/{TARGET_SUB_300_COLORINGS:4d} | "
        f"elapsed={elapsed:8.1f}s"
        f"{flag_text}"
    )

In [ ]:
# Populate the Sub-300 Pool

batch_start = perf_counter()

batch_result = batch.populate(
    batch_config,
    observer=report_attempt,
)

batch_elapsed = perf_counter() - batch_start

print()
print("Attempts completed:", batch_result.attempts_completed)
print("Target reached:", batch_result.target_reached)
print("Initial eligible:", batch_result.initial_eligible_count)
print("Final eligible:", batch_result.final_eligible_count)
print("New eligible:", batch_result.new_eligible_colorings)
print("Best attempt score:", batch_result.best_score)
print("Archive best:", archive.best_score(graph))
print("Elapsed:", f"{batch_elapsed:.3f} seconds")

if batch_result.attempts_completed:
    print(
        "Mean time per attempt:",
        f"{batch_elapsed / batch_result.attempts_completed:.3f} seconds",
    )

In [ ]:
# Sub-300 Archive Summary

score_bands = (
    (0, 99),
    (100, 149),
    (150, 174),
    (175, 199),
    (200, 249),
    (250, 299),
)

print("Archive best:", archive.best_score(graph))

for minimum_score, maximum_score in score_bands:
    count = archive.coloring_count_in_score_range(
        minimum_score=minimum_score,
        maximum_score=maximum_score,
        graph=graph,
    )

    print(
        f"Scores {minimum_score:3d}–{maximum_score:3d}:",
        count,
    )

print(
    "Total sub-300:",
    archive.coloring_count_in_score_range(
        maximum_score=299,
        graph=graph,
    ),
)

In [ ]:
# Release the SQLite Connection

archive.close()
print("Archive closed.")